# 7. Capstone: Mini AI Harness

One runtime now hosts a research agent and a safe task agent. Demonstrate configuration loading, scoped discovery, validation, policy, bounded execution, events, approval/checkpoint resume, memory interface, and MCP discovery.

## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Host two agent configurations with one live/mock runtime, registry, policy, memory, events, checkpoints, and MCP boundary.

Architecture reference: [Day 5 diagrams D16–D18](../../diagrams/source/day_05.md).

### Expected observation

Research completes with evidence; task sending pauses; the explicit resume decision determines the outcome. Exact IDs, timing, and live wording will vary.

## Concept briefing

## Mapping the course to production systems

| Course term | Common production terminology |
|---|---|
| Provider adapter | model client/provider layer |
| Agent configuration | agent definition/profile |
| Harness runtime | agent runtime/orchestration layer |
| Tool registry | tool/plugin registry |
| Policy | authorization or guardrail middleware |
| Events | tracing/telemetry |
| Checkpoint store | durable execution/state persistence |
| MCP client | protocol integration layer |

Production SDKs package different subsets of these responsibilities. Students should be
able to open an unfamiliar SDK and locate where its model calls, tools, policy, state and
events live rather than assuming the SDK itself is the architecture.

## What the mini harness does not provide

The classroom harness is intentionally not a production platform. It does not provide
enterprise identity, operating-system sandboxing, remote MCP authentication, distributed
workers, deployment or guaranteed model quality. Its purpose is to make the essential
boundaries visible so students can recognise and evaluate larger systems later.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
runtime=HarnessRuntime(build_demo_registry(),MockModel())
cases=[("research_agent","What is a harness?"),("task_agent","Prepare a concise project update")]
for name,prompt in cases:
    result=runtime.run(load_config(name),prompt)
    print(name,result.status,result.output)
    print("events:",[e["event"] for e in result.events])

In [ ]:
task=load_config("task_agent")
pending=runtime.run(task,"Send a synthetic project update")
assert pending.status=="pending_approval"
print("Approval card:",pending.pending_action)
final=runtime.resume(pending.run_id,task,approved=True)
print(final.status,final.output)
print([e["event"] for e in final.events])

In [ ]:
memory=SimpleMemory(); memory.add("fictional_asha","Prefer concise project updates")
print(memory.search("fictional_asha","concise update"))
async def mcp_check():
    client=FakeMCPClient(); return await client.list_tools(),await client.call_tool("course_lookup",{"topic":"harness"})
tools,mcp_result=await mcp_check(); print(tools,mcp_result)

## Final explanation

Draw: **agent config → runtime → provider/tool proposal → registry validation → policy → approval or execution → events/checkpoint**. MCP enters through discovery/invocation but still passes local policy.

Defend what this harness does *not* provide: authentication, OS sandboxing, remote MCP trust, distributed workers, deployment, or guaranteed model quality. Optional next steps are FastAPI, Docker, SQLite checkpoints, LangSmith/OpenTelemetry export, and a second hosted provider—not core requirements.

## Your turn

Add a third configuration without modifying runtime.py and submit its event trace plus one denied action.

## Recap

The harness is reusable infrastructure, not a universal autonomous agent. Name one responsibility that deliberately remains application-specific.